# Data Preparation: Train Data
## Version 2

Two datasets are used:

- [Myanmar ALT](https://www2.nict.go.jp/astrec-att/member/mutiyama/ALT/my-alt-190530.zip) from [ALT Treebank Corpus](https://www2.nict.go.jp/astrec-att/member/mutiyama/ALT/)
- [Sayar's myPos version 3.0](https://github.com/ye-kyaw-thu/myPOS/blob/master/corpus-ver-3.0/corpus/mypos-ver.3.0.txt)

These datasets are then preprocessed and normalized into a standardized format for the language model (LM), primarily for KenLM.

## Setup

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
ROOT = Path("..").resolve()

TRAIN_DIR = ROOT / "data" / "train"
LOG_DIR = TRAIN_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_TEXT = ROOT / "clean_text.py"

SYL_NORM = ROOT / "syl-normalizer" / "syl_normalizer.py"
SYL_DICT = ROOT / "syl-normalizer" / "final_syl_dictionary_13Feb2024.sorted.txt"

OPPA_WORD = ROOT / "oppaword" / "oppa_word.py"
OPPA_DICT = ROOT / "oppaword" / "myg2p_mypos.dict"

## Load All Datasets

In [3]:
!pwd

/home/lawun330/Desktop/burmese-domain-specific-lm/notebooks


In [4]:
df1 = pd.read_csv(
    f"{TRAIN_DIR}/my-alt-190530-data",
    sep="\t",
    header=None,
    names=["id", "sentence"]
)
df1.head()

,id,sentence
0,SNT.80188.1,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,SNT.80188.2,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,SNT.80188.3,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,SNT.80188.4,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,SNT.80188.5,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


In [5]:
with open(f"{TRAIN_DIR}/mypos-ver.3.0.shuf.notag.nopunc.txt", encoding="utf-8") as f:
    lines = f.read().splitlines()
df2 = pd.DataFrame({"text": lines})
df2.head()

,text
0,၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရ...
1,လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထ...
2,ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည်
3,စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ...
4,ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ


### Remove Uncessary Data

- The first dataset contains an unnecessary column: `id`.

In [6]:
df1_new = df1.drop(columns=["id"])
df1_new = df1_new.rename(columns={"sentence": "text"})  # optional
df1_new.head()

,text
0,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


## Merge All Datasets

In [7]:
print(df1_new.shape)
print(df2.shape)

(20106, 1)
(43196, 1)


In [8]:
df = pd.concat([df1_new, df2], ignore_index=True)
df.head()

,text
0,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


In [9]:
print(df.shape)

(63302, 1)


In [10]:
df["text"].to_csv(f"{TRAIN_DIR}/combined_2.raw", index=False, header=False, lineterminator="\n")

## Preprocess Merged Dataset

### Remove Punctuations and Word Tags

The goal of this project is to use KenLM. Therefore, all annotations are removed for now to create a standard corpus.

In [11]:
!python {CLEAN_TEXT} -i {TRAIN_DIR}/combined_2.raw -o {TRAIN_DIR}/combined_2.cleaned.state1

Success! Cleaned text saved to: /home/lawun330/Desktop/burmese-domain-specific-lm/data/train/combined_2.cleaned.state1


### Syllable Normalization

In [12]:
!python {SYL_NORM} \
    --dictionary {SYL_DICT} \
    --frequency 2 \
    --input {TRAIN_DIR}/combined_2.cleaned.state1 \
    --output {TRAIN_DIR}/combined_2.cleaned.state2 \
    --log {TRAIN_DIR}/logs/combined_2.cleaned.state2.log \
    --error-output ../data/train/logs/combined_2.cleaned.state2.errors \
    --fuzzy-distance 0

  Fuzzy correction: disabled

=== syl_normalizer summary (v0.6) ===
  Lines processed    :     63,302
  Tokens processed   :  1,192,846
  Passthrough        :     22,055
  Already valid      :    723,673

  Fixed - stage 1 (rules, token count)    :    1,440
  Fixed - stage 1 (individual rule apps)  :    1,440

  Fixed - stage 2 ngram (dup vowel)       :        0
  Fixed - stage 2 ngram (asat removal)    :        0
  Fixed - stage 2 ngram (char sub)        :        0
  Fixed - stage 2 ngram (medial order)    :        0
  Fixed - stage 2 ngram (dup+asat)        :        0

  Fixed - stage 2 dict (dup vowel)        :        0
  Fixed - stage 2 dict (asat removal)     :        0
  Fixed - stage 2 dict (char sub)         :        0
  Fixed - stage 2 dict (medial order)     :        0
  Fixed - stage 2 dict (dup+asat)         :        0

  Fixed - stage 3 (merge with previous)   :        1
  Fixed - stage 4 (compound split)        :  378,181

  Total fixed        :    379,622
  Still unknown

### Word Segmentation (OppaWord)

In [13]:
!python {OPPA_WORD} \
  --input {TRAIN_DIR}/combined_2.cleaned.state2 \
  --dict {OPPA_DICT} \
  --output {TRAIN_DIR}/combined_2.cleaned.state3

## Load Final Dataset

In [14]:
new_df = pd.read_csv(f"{TRAIN_DIR}/combined_2.cleaned.state3", header=None, names=["text"])
new_df

,text
0,ပြင် သစ် နိုင် ငံ ပါ ရီ မြို့ ပါ့ ဒက်စ် ပ ရ င့...
1,အန် ဒ ရီ ယာ မာ စီ သည် အီ တ လီ အ တွက် စမ်း သပ် ...
2,ပ ထ မ တစ် ဝက် ၏ တော် တော် များ များ အ တွက် က စ...
3,ပေါ် တူ ဂီ သည် ဘယ် သော အ ခါ မှ စွ န့် လွှတ် မှ...
4,အီ တ လီ သည် ပ ထ မ ပိုင်း ၌ ၁၆ ၅ ဖြ င့် ဦး ဆောင...
...,...
63297,အ ခု ဘာ လုပ် နေ လဲ
63298,ဇူ လိုင် ၁၄ ရက် မှာ ဘန် ကောက် ကို သွား မ ယ့် U...
63299,ကား မှ ကား ဘီး ကို ဖြုတ် လိုက် သည်
63300,ကျွန် တော် သိ ပါ ရ စေ


## Exploring the Final Dataset File Size

In [15]:
!ls -lh {TRAIN_DIR}/combined_2.raw {TRAIN_DIR}/combined_2.cleaned.state3

-rw-rw-r-- 1 lawun330 lawun330 18M May 16 00:19 /home/lawun330/Desktop/burmese-domain-specific-lm/data/train/combined_2.cleaned.state3
-rw-rw-r-- 1 lawun330 lawun330 27M May 16 00:18 /home/lawun330/Desktop/burmese-domain-specific-lm/data/train/combined_2.raw


## References
- ```bibtex
  @misc{syl_normalizer,
    author       = {Ye Kyaw Thu},
    title        = {{Syllable Normalization Tool for Myanmar Language}},
    version      = {0.6},
    month        = {May},
    year         = {2026},
    publisher    = {GitHub},
    url          = {https://github.com/ye-kyaw-thu/syl-Normalizer/tree/main/ver_0.6},
    note         = {Accessed: YYYY-MM-DD},
    institution  = {Language Understanding Lab (LU Lab), Myanmar}
  }
  ```

- ```bibtex
  @misc{oppaWord_2025,
    author       = {Ye Kyaw Thu},
    title        = {{oppaWord: Hybrid DAG+Bi-MM+LM Myanmar Word Segmenter}},
    version      = {1.0},
    month        = {August},
    year         = {2025},
    publisher    = {GitHub},
    url          = {https://github.com/ye-kyaw-thu/oppaWord},
    note         = {Accessed: YYYY-MM-DD},
    institution  = {Language Understanding Lab (LU Lab), Myanmar}
  }
  ```